In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import *
from delta.tables import *
from pyspark.sql import Window

In [0]:
SourceSchemaPath='/mnt/raw/SchemaMapping/SchemaMappingNew.csv'
targetSchemaPath='/mnt/silver/DatabaseSchema'

In [0]:
Schema= StructType([
    # StructField('SchemaID', IntegerType()),
    StructField('DeviceType',StringType()),
    StructField('TableName', StringType()),
    StructField('LogType', StringType()),
    StructField('ContractName', StringType()),
    StructField('SourceName',StringType(),True),
    StructField('TargetName', StringType()),
    StructField('DataType', StringType()),
    StructField('ColumnType', StringType()),
    StructField('updateEnabled', StringType()),
    # StructField('CreatedDate', TimestampType()),
    # StructField('CreatedBy', StringType()),
    # StructField('UpdatedDate', TimestampType()),
    # StructField('UpdatedBy', StringType()),
    StructField('isActive', StringType())
])

In [0]:
user_id = spark.sql('select current_user() as user').collect()[0]['user']
print(user_id)

In [0]:
SchemaMapping=(spark.read.format('csv').option("header","True")
               .option("delimiter",",").schema(Schema)
                .load(SourceSchemaPath))
                
SchemaMapping=(SchemaMapping.withColumn("DeviceType",split(col("DeviceType"),","))
               .withColumn("LogType",split(col("LogType"),","))
               .withColumn("SourceName",split(col("SourceName"),","))
               .withColumn("CreatedDate",current_timestamp())
               .withColumn("UpdatedDate",current_timestamp())
               .withColumn("CreatedBy", lit(user_id))
               .withColumn("updatedBy", lit(user_id))
               .withColumn("updatedBy", lit(user_id))
               .withColumn("SchemaID",row_number().over(Window.orderBy("TableName")))
             )
SchemaMapping.display()

In [0]:
SchemaMapping.write.format('delta').option("mergeSchema","true").mode('overwrite').save(targetSchemaPath)

In [0]:
%sql
drop table if Exists SilverZone.DatabaseSchema;
create table SilverZone.DatabaseSchema using delta location '/mnt/silver/DatabaseSchema'

In [0]:
%sql
select * from silverzone.databaseschema 

In [0]:
dbutils.notebook.exit("done")